# RetinoCheck AI — ResNet-18 training

**Research only. Never a medical device or medical advice.**

Trains `resnet18` on the APTOS 2019 fundus dataset with the canonical settings
defined in `src/config.py`.  Settings are identical to `04_train_efficientnet_b0.ipynb`
so the two runs are directly comparable.

**Before running:** select *Runtime › Change runtime type › T4 GPU*, then run
the setup cell.  Replace `REPO_URL` with your GitHub clone URL.

## 1 · Environment setup
Mounts Drive, clones the repo (or reuses an existing clone), installs pinned
packages, and verifies CUDA is available.  Run once per Colab session.

In [ ]:
# ── Replace with your fork URL before running ──────────────────────────────
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
# ───────────────────────────────────────────────────────────────────────────
from google.colab import drive
import os
from pathlib import Path
import subprocess
import sys

REPO_DIR  = Path("/content/retinocheck-ai")
DRIVE_ROOT = Path("/content/drive/MyDrive/RetinoCheckAI")

if "YOUR_USERNAME" in REPO_URL or "YOUR_REPO" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository URL first.")
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("These pins target Python 3.11–3.13. Select a matching Colab runtime.")

drive.mount("/content/drive")
for folder in ("checkpoints", "reports"):
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git clone. Choose another REPO_DIR.")
else:
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != REPO_URL:
        raise RuntimeError(f"Existing clone has a different origin: {remote}")
    print("Using existing clone. Commit:", subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# CUDA 12.4 wheels — the official matching PyTorch 2.6 / torchvision 0.21 pair.
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch==2.6.0", "torchvision==0.21.0",
                "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

# Verify in a subprocess so newly installed versions are visible.
subprocess.run([sys.executable, "-c", """
import torch, torchvision, timm, datasets, sklearn, gradio, pytorch_grad_cam
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. Select a T4 GPU runtime and rerun setup.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU tensor check:', (torch.ones(1, device='cuda') + 1).item())
"""], check=True)
print("Setup complete. Restart the kernel if packages were already imported before this cell.")

## 2 · Imports and configuration

All hyperparameters live in `Config`.  The only difference between this
notebook and `04_train_efficientnet_b0.ipynb` is `MODEL = "resnet18"`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from src.config import Config, CHECKPOINT_ROOT
from src.data import load_source
from src.train import train
from src.evaluate import evaluate

# ── Model selection (only line that differs from notebook 04) ───────────────
MODEL = "resnet18"
# ───────────────────────────────────────────────────────────────────────────

# Canonical settings shared with efficientnet_b0 for a fair comparison.
config = Config(
    model_name=MODEL,
    output_dir=str(CHECKPOINT_ROOT / MODEL),
    mode="classification",   # switch to "regression" for ordinal head
    num_classes=5,
    epochs=15,
    lr=1e-4,
    weight_decay=1e-2,
    batch_size=32,
    image_size=224,
    ben_graham=True,
    class_weights=True,
    use_weighted_sampler=False,
    patience=5,
    seed=42,
    num_workers=2,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model : {MODEL}")
print(f"Mode  : {config.mode}")
print(f"Device: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU — select a T4 runtime'}")
print(f"Output: {config.output_dir}")

## 3 · Data verification
Loads the dataset, checks the schema and all five grade labels, and decodes
one image to catch broken dependencies before training starts.

In [ ]:
source, labels, identity = load_source(config)
print(f"Images   : {len(labels):,}")
print(f"Fingerprint: {identity}")
print("Grade counts:", {int(g): int((labels == g).sum()) for g in range(5)})
del source  # free RAM; make_loaders re-opens the dataset during training

## 4 · Train (or resume after disconnection)

Set `RESUME = True` after a Colab disconnect: the cell reuses the last
completed epoch from `last.pt` on Drive and continues seamlessly.
The scheduler and RNG state are restored so the LR curve is continuous.

> **Do not** set `RESUME = True` for a different experiment — use a new
> `output_dir` instead.

In [ ]:
RESUME = False  # ← flip to True after a Colab disconnect

last_ckpt = Path(config.output_dir) / "last.pt"
resume_path = str(last_ckpt) if RESUME and last_ckpt.exists() else None

if RESUME and not last_ckpt.exists():
    raise FileNotFoundError(
        f"No checkpoint found at {last_ckpt}.\n"
        "Run the cell with RESUME=False to start a fresh training run."
    )

history = train(config, resume=resume_path)

## 5 · Training curves
Reads the log written by `train()` so the plot is available even after a
kernel restart (just re-run this cell and the ones below without training again).

In [ ]:
log_path = Path(config.output_dir) / "history.json"
history = json.loads(log_path.read_text(encoding="utf-8"))

epochs     = [r["epoch"]      for r in history]
train_loss = [r["train_loss"] for r in history]
val_loss   = [r["val_loss"]   for r in history]
qwk        = [r["qwk"] if r["qwk"] is not None else float("nan") for r in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"ResNet-18 training curves (mode={config.mode})", fontsize=13)

# --- Loss ---
ax1.plot(epochs, train_loss, marker="o", label="train loss")
ax1.plot(epochs, val_loss,   marker="s", label="val loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Train / Val loss")
ax1.legend()
ax1.grid(alpha=0.3)

# --- QWK ---
best_epoch = epochs[qwk.index(max(qwk))]
best_qwk   = max(qwk)
ax2.plot(epochs, qwk, marker="D", color="tab:green", label="val QWK")
ax2.axvline(best_epoch, color="tab:green", linestyle="--", alpha=0.6,
            label=f"best epoch {best_epoch} (QWK={best_qwk:.4f})")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Quadratic Weighted Kappa")
ax2.set_title("Validation QWK")
ax2.legend()
ax2.grid(alpha=0.3)

fig.tight_layout()
plot_path = Path(config.output_dir) / "training_curves.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Curves saved → {plot_path}")

## 6 · Numeric training summary
Prints the best and final epoch, and the paths of the two checkpoints written
to Drive.  Use these paths in `05_evaluate.ipynb` for the held-out test set.

In [ ]:
best_row  = max(history, key=lambda r: r["qwk"] or -1)
final_row = history[-1]

ckpt_dir  = Path(config.output_dir)
best_path = ckpt_dir / "best.pt"
last_path = ckpt_dir / "last.pt"
csv_path  = ckpt_dir / "history.csv"

print("=" * 60)
print(f"Model        : {MODEL}")
print(f"Mode         : {config.mode}")
print(f"Epochs run   : {len(history)} / {config.epochs}")
print()
print(f"Best epoch   : {best_row['epoch']}")
print(f"  val QWK    : {best_row['qwk']:.4f}")
print(f"  val loss   : {best_row['val_loss']:.4f}")
print(f"  accuracy   : {best_row['accuracy']:.4f}")
print(f"  macro-F1   : {best_row['macro_f1']:.4f}")
print()
print(f"Final epoch  : {final_row['epoch']}")
print(f"  val QWK    : {final_row['qwk']:.4f}")
print(f"  train loss : {final_row['train_loss']:.4f}")
print()
print(f"Best ckpt    : {best_path}")
print(f"Last ckpt    : {last_path}")
print(f"CSV log      : {csv_path}")
print("=" * 60)

## 7 · Save environment
Pins the exact package versions alongside the checkpoint so the run is
reproducible even if upstream packages change.

In [ ]:
import subprocess
from pathlib import Path

env_path = Path(config.output_dir) / "environment.txt"
env_path.write_text(
    subprocess.check_output(["pip", "freeze"], text=True), encoding="utf-8"
)
print(f"Environment saved → {env_path}")

## 8 · Held-out test evaluation

> **Run this cell once only**, after you have selected the final model using
> validation QWK.  Do not use test results to tune hyperparameters.

In [ ]:
best_ckpt   = str(Path(config.output_dir) / "best.pt")
report_dir  = str(Path(config.output_dir) / "reports")

test_results = evaluate(best_ckpt, output_dir=report_dir, split="test")
print("Test QWK  :", test_results["qwk"])
print("Accuracy  :", test_results["accuracy"])
print("Macro-F1  :", test_results["macro_f1"])
print(f"Full report → {report_dir}")

---
## 9 · Regularisation comparison experiment

### Theory: L1 vs L2 geometry

Both penalties shrink weights toward zero, but their constraint sets have
very different shapes — and that shape determines whether sparsity emerges.

**L2 (weight decay, AdamW)**  
The feasible region is a *sphere* (circle in 2-D):
$\{w : \|w\|_2^2 \le C\}$.  
Every point on the sphere boundary is equally smooth — the minimum of the
task loss almost never hits a corner, so weights are *shrunk* but rarely
driven to exactly zero.  AdamW implements this efficiently by subtracting
$\eta \lambda w$ directly from the weight at each step, decoupled from the gradient.

**L1 (manual penalty)**  
The feasible region is a *diamond* (rhombus in 2-D):
$\{w : \|w\|_1 \le C\}$.  
Its boundary has sharp corners sitting exactly on the coordinate axes.
The task-loss contour is very likely to first touch the boundary at a corner,
which sets one or more weights to exactly zero — this is *sparsity*.
We add the penalty directly to the forward-pass loss so autograd differentiates
through $\lambda \sum_i |w_i|$, making every gradient include a sign-based
shrinkage term $\pm\lambda$.

**When to prefer which**

| | L2 (AdamW) | L1 (manual) |
|---|---|---|
| Geometry | sphere | diamond |
| Effect | uniform shrinkage | sparsity (zero weights) |
| Feature selection | no | yes (implicitly) |
| Typical use | almost always | interpretability / compression |

The three arms below isolate the effect by sharing every other hyperparameter
and the same random seed:

| Arm | `weight_decay` | `l1_lambda` | Description |
|---|---|---|---|
| **(a) None** | 0 | 0 | No regularisation — baseline |
| **(b) L2**   | 1e-2 | 0 | AdamW weight decay only |
| **(c) L1**   | 0 | 1e-5 | Manual L1 penalty only |


In [ ]:
# ── Re-import if running this section after a kernel restart ──────────────────
import json
from pathlib import Path
import matplotlib.pyplot as plt
import torch
from src.config import Config, CHECKPOINT_ROOT
from src.train import train_one_config

MODEL = "resnet18"

# Base config — every arm shares these settings.
# weight_decay and output_dir are overridden per arm inside train_one_config().
base_config = Config(
    model_name=MODEL,
    output_dir=str(CHECKPOINT_ROOT / MODEL / "reg_experiment"),
    mode="classification",
    num_classes=5,
    epochs=10,          # shorter for the ablation — increase if GPU time allows
    lr=1e-4,
    weight_decay=1e-2,  # overridden per arm
    batch_size=32,
    image_size=224,
    ben_graham=True,
    class_weights=True,
    use_weighted_sampler=False,
    patience=0,         # disable early stopping so every arm runs the same epochs
    seed=42,
    num_workers=2,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"Base output: {base_config.output_dir}")
all_histories = {}  # populated by the next cell; read back from disk if empty


Run all three arms sequentially.  Each arm writes its own `last.pt`, `best.pt`,
`history.json`, and `history.csv` to a sub-directory so results survive a
kernel restart without re-training.


In [ ]:
ARMS = [
    dict(output_subdir="arm_a_none",  weight_decay=0.0,  l1_lambda=0.0,  label="(a) None"),
    dict(output_subdir="arm_b_l2",    weight_decay=1e-2, l1_lambda=0.0,  label="(b) L2"),
    dict(output_subdir="arm_c_l1",    weight_decay=0.0,  l1_lambda=1e-5, label="(c) L1"),
]

for arm in ARMS:
    sep = "=" * 55
    print(f"\n{sep}")
    print(f"  Arm {arm['label']}  |  wd={arm['weight_decay']}  l1={arm['l1_lambda']}")
    print(sep)
    hist = train_one_config(base_config, **arm)
    all_histories[arm["label"]] = hist
    best_qwk = max(r["qwk"] or -1 for r in hist)
    print(f"  Done. Best val QWK: {best_qwk:.4f}")

print("\nAll arms complete.")


Re-load results from disk if needed (safe after a kernel restart), then print
the comparison table and the four-panel plot.


In [ ]:
BASE_REG = Path(base_config.output_dir)
SUBDIRS  = {
    "(a) None": "arm_a_none",
    "(b) L2"  : "arm_b_l2",
    "(c) L1"  : "arm_c_l1",
}

# Prefer in-memory results; fall back to JSON logs on Drive.
if not all_histories:
    all_histories = {
        label: json.loads((BASE_REG / sub / "history.json").read_text(encoding="utf-8"))
        for label, sub in SUBDIRS.items()
    }

# ── Comparison table ───────────────────────────────────────────────────────────────────
col_w = 14
headers = ["Arm", "Best val QWK", "Final val QWK", "Train-val gap", "Best epoch"]
print("".join(h.ljust(col_w) if i == 0 else h.rjust(col_w) for i, h in enumerate(headers)))
print("-" * 70)
table_rows = []
for label, hist in all_histories.items():
    best_row  = max(hist, key=lambda r: r["qwk"] or -1)
    final_row = hist[-1]
    gap       = final_row["train_loss"] - final_row["val_loss"]
    best_qwk  = best_row["qwk"] or float("nan")
    final_qwk = final_row["qwk"] or float("nan")
    table_rows.append(dict(label=label, best_qwk=best_qwk, final_qwk=final_qwk,
                           gap=gap, best_epoch=best_row["epoch"]))
    row_vals = [label, f"{best_qwk:.4f}", f"{final_qwk:.4f}",
                f"{gap:.4f}", str(best_row["epoch"])]
    print("".join(v.ljust(col_w) if i == 0 else v.rjust(col_w) for i, v in enumerate(row_vals)))

# ── Four-panel plot ──────────────────────────────────────────────────────────────────
COLORS  = {"(a) None": "tab:gray", "(b) L2": "tab:blue",   "(c) L1": "tab:orange"}
MARKERS = {"(a) None": "o",        "(b) L2": "s",           "(c) L1": "D"}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("ResNet-18 regularisation comparison  (a: None · b: L2 weight-decay · c: L1 penalty)",
             fontsize=11)

ax_tl, ax_vl   = axes[0, 0], axes[0, 1]
ax_qwk, ax_gap = axes[1, 0], axes[1, 1]

for label, hist in all_histories.items():
    ep  = [r["epoch"]      for r in hist]
    tr  = [r["train_loss"] for r in hist]
    vl  = [r["val_loss"]   for r in hist]
    qwk = [r["qwk"] if r["qwk"] is not None else float("nan") for r in hist]
    gap = [t - v for t, v in zip(tr, vl)]
    kw  = dict(color=COLORS[label], marker=MARKERS[label], label=label)
    ax_tl.plot(ep, tr,  **kw)
    ax_vl.plot(ep, vl,  **kw)
    ax_qwk.plot(ep, qwk, **kw)
    ax_gap.plot(ep, gap, **kw)

ax_tl.set(title="Train loss",     xlabel="Epoch", ylabel="Loss")
ax_vl.set(title="Val loss",       xlabel="Epoch", ylabel="Loss")
ax_qwk.set(title="Val QWK",       xlabel="Epoch", ylabel="QWK")
ax_gap.set(title="Train-val gap", xlabel="Epoch", ylabel="train_loss - val_loss")
ax_gap.axhline(0, color="black", linewidth=0.8, linestyle="--")

for ax in axes.flat:
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

fig.tight_layout()
plot_path = BASE_REG / "regularisation_comparison.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot saved \u2192 {plot_path}")
